In [ ]:
from pyiceberg.catalog import load_catalog
import pandas as pd
import pyarrow as pa

catalog = load_catalog(
    "default",
    **{
        "type": "hive",
        "uri": "thrift://hive-metastore:9083",  # Lebih rapi pakai hostname
        "s3.endpoint": "http://minio:9000",      # Lebih rapi pakai hostname
        "s3.access-key-id": "minioadmin",
        "s3.secret-access-key": "minioadmin",
        "s3.use-ssl": "false",
    }
)

: 

In [32]:
catalog.list_namespaces()

[('default',)]

In [33]:
# 1. Cek daftar namespace yang sudah ada
namespaces = catalog.list_namespaces()
print(f"Namespace saat ini: {namespaces}")

# 2. Buat namespace 'bronze' jika belum ada
tgt_namespace = "bronze"
loc_namespace = f"s3a://lakehouse/{tgt_namespace}/"

if any(tgt_namespace in ns for ns in namespaces):
    print(f"ℹ️ Namespace '{tgt_namespace}' sudah tersedia.")
else:
    catalog.create_namespace(tgt_namespace, properties={"location": loc_namespace})
    print(f"✅ Namespace '{tgt_namespace}' berhasil dibuat.")

Namespace saat ini: [('default',)]
✅ Namespace 'bronze' berhasil dibuat.


In [34]:
# # Nama schema yang ingin dibuat
# schema_name = "bronze"
# # Lokasi fizikal di MinIO (pastikan diakhiri dengan /)
# schema_location = "s3a://lakehouse/bronze/"


# # Cara membuat namespace dengan properties location
# try:
#     catalog.create_namespace(
#         schema_name, 
#         properties={"location": schema_location}
#     )
#     print(f"✅ Schema '{schema_name}' berhasil dibuat di {schema_location}")
# except Exception as e:
#     print(f"❌ Gagal atau Schema mungkin sudah ada: {e}")

In [35]:
import pandas as pd
import pyarrow as pa

df = pd.read_parquet("../data/yellow_tripdata_2023-01.parquet")
arrow_table = pa.Table.from_pandas(df)

namespaces = "bronze" # namespace = schema = database
table_name = "yellow_taxi"
table_id = f"{namespaces}.{table_name}"  # Menjadi 'bronze.yellow_taxi'

# Lokasi fisik di MinIO
table_location = f"s3a://lakehouse/{namespaces}/{table_name}"

In [36]:
try:
    # 1. Coba buat tabel baru
    table = catalog.create_table(
        identifier=table_id,
        schema=arrow_table.schema,
        location=table_location
    )
    print(f"✅ Tabel {table_id} berhasil dibuat baru.")
except Exception:
    # 2. Jika sudah ada, load tabel yang sudah ada
    table = catalog.load_table(table_id)
    print(f"ℹ️ Tabel {table_id} sudah ada, memuat tabel...")

# 3. Masukkan data (Append)
table.append(arrow_table)
print(f"🚀 Data berhasil dimuat ke {table_id}!")

✅ Tabel bronze.yellow_taxi berhasil dibuat baru.
🚀 Data berhasil dimuat ke bronze.yellow_taxi!


In [37]:
# import s3fs

# # Inisialisasi koneksi ke MinIO
# fs = s3fs.S3FileSystem(
#     key='minioadmin',
#     secret='minioadmin',
#     client_kwargs={'endpoint_url': 'http://minio:9000'}
# )

# # Hapus folder yang tertinggal
# path_lama = "warehouse/test_schema.yellow_taxi"
# if fs.exists(path_lama):
#     fs.rm(path_lama, recursive=True)
#     print(f"✅ Folder {path_lama} berhasil dibersihkan dari MinIO.")